In [ ]:
!pip install -qU langchain-openai langchain-core tiktoken
!pip install langgraph langchain-groq langchain-core
!pip install mcp langchain-mcp-adapters uvicorn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.3/508.3 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.1 MB/s eta 0:00:00


In [ ]:
import os
import time
from google.colab import userdata
# from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI

# Set your API Key in Colab Secrets (Left sidebar -> Key icon)
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
llm_OpenAI = ChatOpenAI(model="gpt-4o", temperature=0)

GROQ_API_KEY = "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"
MODEL_NAME   = "llama-3.3-70b-versatile"
llm = ChatGroq(model=MODEL_NAME, api_key=GROQ_API_KEY)


# The "Golden Input" - Constants for the experiment
GOLDEN_INPUT = {
    "client_name": "Mr. Jameson",
    "risk_profile": "Moderate-Aggressive",
    # "portfolio_weights": {"NVDA": "25%", "AAPL": "15%", "MSFT": "20%", "TSLA": "10%", "CASH": "30%"},
    "metrics": {"Sharpe_Ratio": 0.82, "Annual_Volatility": "14.2%"},
    "research_insights": [
        "NVDA: Blackwell chip production yields are higher than expected.",
        "AAPL: Services revenue saw a 12% jump but hardware sales are flat in China.",
        "Market Sentiment: Bullish on AI infrastructure, Neutral on consumer electronics."
    ]
}



In [ ]:
def measure_variant(variant_name, variant_func, data):
    start_time = time.time()

    # Execute the variant
    response = variant_func(data)

    end_time = time.time()

    # Extract Metadata
    latency = end_time - start_time
    tokens = response.usage_metadata  # LangChain 2026 standard

    return {
        "Variant": variant_name,
        "Latency": f"{latency:.2f}s",
        "Prompt_Tokens": tokens['input_tokens'],
        "Completion_Tokens": tokens['output_tokens'],
        "Total_Tokens": tokens['total_tokens'],
        "Content": response.content
    }

###Variant A: Pure Prompt

In [ ]:
def variant_a_pure_prompt(data):
    system_instruction = """
    You are an expert Wealth Manager. Generate a professional 1-page report.

    STRUCTURE:
    1. Executive Summary
    2. Portfolio Allocation (Table format)
    3. Quantitative Risk Analysis
    4. Market Research & Commentary

    RULES:
    - Use a professional, confident tone.
    - Reference specific data points provided.
    - Keep it under 500 words.
    """

    user_content = f"""
    Generate a report for {data['client_name']} using this data:
    Risk Profile: {data['risk_profile']}
    Metrics: {data['metrics']}
    Insights: {data['research_insights']}
    """

    messages = [
        SystemMessage(content=system_instruction),
        HumanMessage(content=user_content)
    ]

    return llm.invoke(messages)

In [ ]:
# --- EXECUTION ---
# result_a = measure_variant("Variant A: Pure Prompt", variant_a_pure_prompt, GOLDEN_INPUT)

###Variant B: Tool Calling

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
# Create the MCP Report Generation server
SERVER_CODE = """
from mcp.server.fastmcp import FastMCP
import json

mcp = FastMCP("WealthReportServer", host="127.0.0.1", port=8001)

@mcp.tool()
def generate_allocation_table(weights_json: str) -> str:
    \"\"\"Converts a JSON string of portfolio weights into a clean Markdown table for the report.\"\"\"
    try:
        weights = json.loads(weights_json)
        table = "| Asset Ticker | Targeted Weight |\\n| :--- | :--- |\\n"
        for ticker, weight in weights.items():
            table += f"| {ticker} | {weight} |\\n"
        return table
    except Exception as e:
        return f"Error formatting table: {str(e)}"

@mcp.tool()
def format_market_insights(insights_json: str, sentiment_score: float) -> str:
    \"\"\"Formats raw research strings and a sentiment score into a professional narrative block.\"\"\"
    try:
        insights = json.loads(insights_json)
        sentiment_label = "Bullish" if sentiment_score > 0.3 else "Bearish" if sentiment_score < -0.3 else "Neutral"

        header = f"### Market Analysis & Sentiment ({sentiment_label})\\n"
        body = "\\n\\n".join([f"- {item}" for item in insights])
        return f"{header}\\n{body}"
    except Exception as e:
        return f"Error formatting insights: {str(e)}"

@mcp.tool()
def get_compliance_disclaimer(risk_profile: str) -> str:
    \"\"\"Retrieves the mandatory legal disclaimer based on the client's specific risk profile.\"\"\"
    disclaimers = {
        "Moderate-Aggressive": "STRATEGY DISCLOSURE: This allocation involves high-growth equity targets. Market volatility may impact principal value.",
        "Conservative": "STRATEGY DISCLOSURE: Focus is on capital preservation. Yields may not outpace inflation in all market cycles.",
        "Moderate": "STRATEGY DISCLOSURE: Balanced approach using both growth and value assets."
    }
    content = disclaimers.get(risk_profile, "Standard financial risk disclosure applies.")
    return f"\\n---\\n**Compliance Notice:** {content}"

if __name__ == "__main__":
    # Note: FastMCP uses SSE transport for external communication
    mcp.run(transport="sse")
"""

with open("report_server.py", "w") as f:
    f.write(SERVER_CODE.strip())

print("report_server.py written (SSE transport, port 8001).")

report_server.py written (SSE transport, port 8001).


In [ ]:
import subprocess
import time

# Launch the MCP server as a background subprocess.
# It listens on http://127.0.0.1:8001/sse via SSE transport.
server_proc = subprocess.Popen(
    ["python", "report_server.py"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)  # allow uvicorn to start
print(f"MCP server started (PID: {server_proc.pid})")

# Connect via SSE transport
mcp_client = MultiServerMCPClient({
    "report_server": {
        "url": "http://127.0.0.1:8001/sse", # Corrected port from 8000 to 8001
        "transport": "sse",
    }
})

# mcp_report_tools = await mcp_client.get_tools()
# mcp_tools = await mcp_client.get_tools()
# print("MCP server connected. Tools available:")
# for t in mcp_tools:
#     print(f"  · {t.name:20} — {t.description}")

MCP server started (PID: 1197)


In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder # Added import
import json

async def variant_b_react_mcp(data):
    """
    Variant B: MCP Tool-Calling via LangGraph ReAct Agent.
    Uses the built-in ReAct loop to automatically orchestrate tool calls.
    """

    # 1. Get the tools from your already connected mcp_client
    # These must be passed as a list to the agent
    all_tools = await mcp_client.get_tools()

    # 2. Define the System Instructions
    system_prompt_content = (
        "You are a Senior Wealth Report Orchestrator. "
        "Your goal is to build a high-quality 1-page investment report. "
        "STRATEGY: "
        "1. Use 'generate_allocation_table' for the portfolio data. "
        "2. Use 'format_market_insights' for the research bullets. "
        "3. Use 'get_compliance_disclaimer' to finalize the legal section. "
        "Always use these tools rather than generating tables or legal text manually."
    )
    # Correctly integrate the system prompt into a ChatPromptTemplate
    prompt = ChatPromptTemplate.from_messages(
        [
            SystemMessage(content=system_prompt_content),
            MessagesPlaceholder(variable_name="messages"),
        ]
    )

    # 3. Create the ReAct Agent
    # This replaces the manual 'while' loop or double 'invoke' calls
    agent = create_react_agent(
        model=llm,
        tools=all_tools,
        prompt=prompt # Pass the prompt template here
    )

    # 4. Execute the Agent
    # We pass the human query with the raw data
    inputs = {
        "messages": [
            HumanMessage(content=f"Generate a full report for {data['client_name']}. Data: {json.dumps(data)}")
        ]
    }

    # The agent will run as many steps as needed to call all 3 tools
    result = await agent.ainvoke(inputs)

    # 5. Extract the final response from the message history
    # The last message in the state will be the AI's final synthesized report
    final_response = result["messages"][-1]

    return final_response

# --- EXECUTION ---
# result_b = await measure_variant_async("Variant B: ReAct MCP Agent", variant_b_react_mcp, GOLDEN_INPUT)

In [ ]:
async def measure_variant_async(variant_name, variant_func, data):
    start_time = time.perf_counter()

    # Execute the async variant
    response = await variant_func(data)

    end_time = time.perf_counter()

    latency = end_time - start_time
    tokens = response.usage_metadata

    return {
        "Variant": variant_name,
        "Latency": f"{latency:.2f}s",
        "Total_Tokens": tokens['total_tokens'],
        "Content": response.content
    }

In [ ]:
# --- EXECUTION ---
# In Colab, you can run this directly in a cell:
# result_b = await measure_variant_async("Variant B: MCP Server", variant_b_react_mcp, GOLDEN_INPUT)

/tmp/ipykernel_7122/527399401.py:36: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


###Variant C: Agent Skills

In [ ]:
SKILL_DEFINITION = """
# SKILL: Wealth Report Synthesis v2.1
# ROLE: Senior Wealth Communications Officer

## INSTRUCTIONS
1. Analyze the 'Sentiment' from the state.
2. Use the 'ReportServer' MCP tools to generate valid Markdown components.
3. Synthesize the narrative:
    - Tone: Institutional, objective, yet personalized.
    - Focus: Connect the "Why" (Research) to the "What" (Allocation).
4. Validation: Ensure the total weight equals 100% and the Compliance Notice is appended.

## TEMPLATE
# Investment Strategy Report: {{client_name}}
## Executive Summary
{{summary_text}}
... [Detailed sections follow] ...
"""

with open("wealth_report_skill.md", "w") as f:
    f.write(SKILL_DEFINITION.strip())

print("Skill Registry: wealth_report_skill.md created.")

Skill Registry: wealth_report_skill.md created.


In [ ]:
async def variant_c_agent_skills(data):
    """
    Variant C: AgentSkills (Progressive Disclosure).
    1. Discovery: LLM identifies the need for the 'Wealth Report' skill.
    2. Hydration: System injects the full SKILL.md into the context.
    3. Execution: Agent performs the task with 'Expert-level' instructions.
    """

    # --- PHASE 1: DISCOVERY (Simulated) ---
    # In a real system, the LLM would say "I need the Wealth Report skill."
    # For the experiment, we trigger the 'Hydration' immediately to measure performance.

    # --- PHASE 2: HYDRATION ---
    with open("wealth_report_skill.md", "r") as f:
        expert_instructions = f.read()

    # Fetch tools from MCP (same as Variant B)
    mcp_tools = await mcp_client.get_tools()
    llm_with_skills = llm.bind_tools(mcp_tools)

    # Create a dictionary for easy tool lookup by name
    tools_by_name = {tool.name: tool for tool in mcp_tools}

    # --- PHASE 3: EXECUTION ---
    # Notice how we combine the Skill + Tools + Data only at this specific node
    messages = [
        SystemMessage(content=f"CORE SKILLSET LOADED:\n{expert_instructions}"),
        HumanMessage(content=f"Execute skill for client: {data['client_name']}. Data: {json.dumps(data)}")
    ]

    # Step 1: Agent uses the Skill instructions to orchestrate the MCP tools
    ai_msg = await llm_with_skills.ainvoke(messages)

    # Step 2: Process tool calls
    tool_results = []
    if ai_msg.tool_calls:
        for tc in ai_msg.tool_calls:
            tool_to_call = tools_by_name.get(tc['name'])
            if tool_to_call:
                # result is often a list of content objects in MCP
                result = await tool_to_call.ainvoke(tc['args'])

                # Robustly extract the text regardless of format
                if isinstance(result, list) and len(result) > 0:
                    # If it's the MCP content list format
                    text_content = getattr(result[0], 'text', str(result[0]))
                    tool_results.append(text_content)
                else:
                    # Fallback if it's already a string or different object
                    tool_results.append(str(result))
            else:
                tool_results.append(f"Error: Tool '{tc['name']}' not found.")

    # Step 3: Final Polishing (Driven by the Skill Template)
    final_context = "\n\n".join(tool_results)
    final_report = await llm.ainvoke([
        SystemMessage(content=f"Finalize the report using this Skill Template:\n{expert_instructions}"),
        HumanMessage(content=f"Components:\n{final_context}")
    ])

    return final_report

In [ ]:
# --- EXECUTION ---
# result_c = await measure_variant_async("Variant C: AgentSkills", variant_c_agent_skills, GOLDEN_INPUT)

In [ ]:
# Print Summary Table
# import pandas as pd
# df = pd.DataFrame([result_a, result_b, result_c])
# print(df[["Variant", "Latency", "Total_Tokens"]])
# print("\n--- REPORT OUTPUT --- \n", result_a["Content"])

                  Variant Latency  Total_Tokens
0  Variant A: Pure Prompt   1.88s           618
1   Variant B: MCP Server   1.79s          1096
2  Variant C: AgentSkills   2.17s           799

--- REPORT OUTPUT --- 
 **Wealth Management Report for Mr. Jameson**

### 1. Executive Summary

As your wealth manager, I am pleased to present this report tailored to your moderate-aggressive risk profile. Our analysis indicates a favorable market outlook, with a focus on AI infrastructure and select technology stocks. The current portfolio allocation is positioned to capitalize on these trends while managing risk. Key metrics show a Sharpe Ratio of 0.82, indicating a strong risk-adjusted return, and an Annual Volatility of 14.2%, which is within our expected range for a moderate-aggressive portfolio.

### 2. Portfolio Allocation

| Asset Class | Allocation (%) |
| --- | --- |
| Stocks | 60 |
| Bonds | 30 |
| Alternatives | 10 |
| Cash | 0 |
| **Stocks Breakdown** |  |
| Technology | 40 |
| Heal

In [ ]:
server_proc.terminate()
print("Experiment complete. Server offline.")

Experiment complete. Server offline.


###LLM as a Judge & Final comparison

In [ ]:
import re

def judge_llm_quality(variant_output, original_data):
    """
    Acts as a Senior Wealth Manager to grade the generated report.
    """
    judge_prompt = f"""
    You are a Senior Wealth Management Auditor. Grade the following investment report based on the provided source data.

    ### SOURCE DATA:
    {json.dumps(original_data, indent=2)}

    ### REPORT TO GRADE:
    {variant_output}

    ### GRADING RUBRIC (Score 1-5):
    1. COHERENCE: Is the tone professional and the narrative logical?
    2. COMPLETENESS: Does it include all tickers, weights, the Sharpe Ratio, and research insights?
    3. FORMATTING: Are tables clean and markdown headers used correctly?

    ### OUTPUT FORMAT:
    Return ONLY a JSON object like this:
    {{"coherence": 5, "completeness": 4, "formatting": 5, "feedback": "Brief reason for scores"}}
    """

    # We use the global LLM as the judge
    response = llm_OpenAI.invoke(judge_prompt)

    # Extract JSON from the judge's response
    try:
        # Use regex to find the JSON block in case the LLM adds markdown backticks
        json_match = re.search(r'\{.*\}', response.content, re.DOTALL)
        return json.loads(json_match.group())
    except:
        return {"coherence": 0, "completeness": 0, "formatting": 0, "feedback": "Judge failed to parse."}

In [ ]:
import asyncio
import pandas as pd

async def run_full_experiment():
    variants = [
        ("Variant A: Pure Prompt", variant_a_pure_prompt),
        ("Variant B: ReAct Agent", variant_b_react_mcp),
        ("Variant C: AgentSkills", variant_c_agent_skills)
    ]

    all_results = []

    for name, func in variants:
        print(f"Running {name}...")

        # 1. Measure Latency and Tokens
        start_time = time.perf_counter()
        # Handle both sync and async functions in our loop
        if asyncio.iscoroutinefunction(func):
            response = await func(GOLDEN_INPUT)
        else:
            response = func(GOLDEN_INPUT)
        end_time = time.perf_counter()

        # 2. Get Judge Scores
        scores = judge_llm_quality(response.content, GOLDEN_INPUT)

        # 3. Aggregate Data
        all_results.append({
            "Variant": name,
            "Latency (s)": round(end_time - start_time, 2),
            "Total Tokens": response.usage_metadata['total_tokens'],
            "Coherence": scores['coherence'],
            "Completeness": scores['completeness'],
            "Formatting": scores['formatting'],
            "Overall Quality": round((scores['coherence'] + scores['completeness'] + scores['formatting']) / 3, 2),
        })
        print(f"Content: {response.content}")

    # Create a nice visual table
    report_df = pd.DataFrame(all_results)
    return report_df

# EXECUTE
# final_comparison = await run_full_experiment()
# display(final_comparison)

In [ ]:
final_comparison = await run_full_experiment()
display(final_comparison)

Running Variant A: Pure Prompt...
Content: **Wealth Management Report for Mr. Jameson**

### 1. Executive Summary
As your dedicated Wealth Manager, I am pleased to present this report tailored to your moderate-aggressive risk profile. Our analysis indicates a favorable outlook for your portfolio, with a Sharpe Ratio of 0.82, suggesting a strong risk-adjusted return. The current annual volatility of 14.2% is within our expected range, given your investment strategy. We will continue to monitor market trends and adjust your portfolio as necessary to optimize performance.

### 2. Portfolio Allocation

| Asset Class | Allocation (%) |
| --- | --- |
| Stocks (Tech) | 40 |
| Stocks (Non-Tech) | 20 |
| Bonds | 30 |
| Alternatives | 10 |

### 3. Quantitative Risk Analysis
Our quantitative risk assessment reveals a Sharpe Ratio of 0.82, indicating a favorable balance between returns and volatility. The annual volatility of 14.2% is manageable, considering your moderate-aggressive risk tolerance

/tmp/ipykernel_620/527399401.py:36: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


Content: Dear Mr. Jameson,

I am pleased to present your personalized 1-page investment report. The report includes a tailored allocation table, research insights, and a compliance disclaimer based on your risk profile.

Below is your allocation table:

| Asset Ticker | Targeted Weight |
| :--- | :--- |
| Stocks | 0.6 |
| Bonds | 0.2 |
| Alternatives | 0.2 |

Our research insights indicate a bullish market sentiment, with key points including:
- NVDA: Blackwell chip production yields are higher than expected.
- AAPL: Services revenue saw a 12% jump but hardware sales are flat in China.
- Market Sentiment: Bullish on AI infrastructure, Neutral on consumer electronics.

Please note the following compliance notice:
**Compliance Notice:** STRATEGY DISCLOSURE: This allocation involves high-growth equity targets. Market volatility may impact principal value.

We hope this report meets your expectations and provides valuable insights into your investment portfolio. If you have any questions o

,Variant,Latency (s),Total Tokens,Coherence,Completeness,Formatting,Overall Quality
0,Variant A: Pure Prompt,1.55,577,5,4,5,4.67
1,Variant B: ReAct Agent,3.58,1124,5,4,5,4.67
2,Variant C: AgentSkills,2.58,789,5,4,5,4.67
